<a href="https://colab.research.google.com/github/honordold/DS1001-LABS-Projects/blob/main/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [25]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item (one item within one order), with its order number and vendor_id carried along.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [26]:
import pandas as pd

# One row per line item: record_path points at the nested 'lines' list,
# and meta copies the order-level fields onto every line row.
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
lines

assert len(lines) == 4, f'expected 4 rows, got {len(lines)}'
assert lines['qty'].sum() == 7, f'expected qty total 7, got {lines["qty"].sum()}'
print(lines)

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [27]:
vendors = pd.json_normalize(vendors_json)

# Left join keeps every line item, even if its vendor isn't in the vendor list.
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
print(joined)

unmatched = joined[joined['_merge'] == 'left_only']
unmatched_vendors = list(unmatched['vendor_id'].unique())
unmatched_units = unmatched['qty'].sum()

print('Unmatched vendor(s):', unmatched_vendors)
print('Units attached to unmatched vendor(s):', unmatched_units)

# Result with this data: no unmatched vendors and 0 units.
# The lab says one vendor should be missing, but V-10 appears in both
# orders_json and vendors_json, so every row comes back as 'both'.
# If a vendor_id were missing from vendors_json, it would show up here
# as 'left_only' with NaN for name and zone.

           item  qty order vendor_id          name zone _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A   both
1          Soda    1     1      V-01  Hoos Burgers    A   both
2   Foam Finger    1     2      V-10     Cav Merch    A   both
3       Hot Dog    3     3      V-01  Hoos Burgers    A   both
Unmatched vendor(s): []
Units attached to unmatched vendor(s): 0


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [28]:
# Choice: show an unmatched vendor as its own row labeled 'UNMATCHED (<vendor_id>)'
# instead of dropping it. Groupby drops NaN keys by default, which would silently
# lose those units. Labeling it keeps the total honest and makes the data problem
# visible to whoever reads the table. (With this data every vendor matched,
# so no UNMATCHED row appears.)
joined['vendor_name'] = joined['name'].fillna('UNMATCHED (' + joined['vendor_id'] + ')')

qty_per_vendor = joined.groupby('vendor_name')['qty'].sum().sort_values(ascending=False)
print(qty_per_vendor)
print('Total:', qty_per_vendor.sum())

vendor_name
Hoos Burgers    6
Cav Merch       1
Name: qty, dtype: int64
Total: 7


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [29]:
# dropna=False keeps a NaN zone group, so units from an unmatched vendor are not silently dropped.
qty_per_zone = joined.groupby('zone', dropna=False)['qty'].sum().sort_values(ascending=False)
print(qty_per_zone)

top_zone = qty_per_zone.idxmax()
print('Zone that sold the most items:', top_zone, 'with', qty_per_zone.max(), 'items')

zone_total = joined.dropna(subset=['zone'])['qty'].sum()
print('Total across known zones:', zone_total)
print('Units with no zone (unmatched vendor):', joined['qty'].sum() - zone_total)

# Zone A sold the most: all 7 items. The zone total is 7, so nothing went missing.
# That's because every vendor matched in Q2. If a vendor had been unmatched, its
# zone would be NaN, a normal groupby would drop those units, and the zone total
# would fall short of 7 by exactly the unmatched units from Q2.

zone
A    7
Name: qty, dtype: int64
Zone that sold the most items: A with 7 items
Total across known zones: 7
Units with no zone (unmatched vendor): 0


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [30]:
import pandas as pd

ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# Build one row per line item. .get() returns a default instead of crashing
# when a key is missing: an order with no 'lines' contributes no rows,
# and a line with no 'qty' gets None.
rows = []
for o in ragged_json:
    for line in o.get('lines', []):
        rows.append({
            'order': o['order'],
            'vendor_id': o['vendor_id'],
            'item': line.get('item'),
            'qty': line.get('qty'),
        })

ragged_lines = pd.DataFrame(rows)

# Int64 (capital I) is pandas' nullable integer type: real quantities stay
# whole numbers and the missing one shows as <NA>, not 0 (no fillna(0)).
ragged_lines['qty'] = pd.array(ragged_lines['qty'].tolist(), dtype='Int64')
ragged_lines['qty_missing'] = ragged_lines['qty'].isna()
print(ragged_lines)

orders_with_no_lines = [o['order'] for o in ragged_json if 'lines' not in o]
print('Orders with no line items:', orders_with_no_lines)
print('Line items missing qty:', int(ragged_lines['qty_missing'].sum()))

   order vendor_id   item   qty  qty_missing
0      4      V-01   Soda     2        False
1      6      V-01  Fries  <NA>         True
Orders with no line items: [5]
Line items missing qty: 1


### Q6 — Validate

In [31]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** Each line item has its own quantity, so one row per line item lets you add up quantities directly. Questions like "how many units did each vendor sell," "how many units were sold in each zone," or "how many Sodas were sold" become a simple groupby and sum. With one row per order, the items and quantities are stuck inside a nested list in a single cell, so you would have to unpack that list every time before doing any math. Order 1 has two different items, and one row per order can't represent that cleanly.

**b)** A quantity of zero means we know the answer: nothing was sold on that line. A missing quantity means we don't know what was sold, because the data didn't tell us. If you fill missing values with 0, unknown data looks like a real zero. Totals then look complete when they are actually undercounted, averages get pulled down, and no one notices the data problem, so it never gets fixed at the source. Keeping it as NA means sums skip it while you can still count and flag how many rows were missing, so the gap stays visible.